# From Waveform to Genre — Analysis & Visualization


**FMA dataset (citation)**  
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is CC BY 4.0; audio files follow per-artist Creative Commons licenses. Consult the original dataset for terms of use.

---

### Project adaptation note
This notebook adapts the FMA data pipeline for the current workflow and interactive environments (Google Colab / Drive). Key adaptations include Colab/Drive path variables, idempotent download & extraction steps, automatic manifest generation for reproducibility, and helper utilities for checksums and seeding. Any reuse of original code or logic from the mdeff/fma project is indicated in the header and documented in the repository.

### Short comparison vs. original mdeff/fma
This notebook focuses on model inference, evaluation and visualization for the trained genre classifier. Compared to the original mdeff/fma repository, this notebook:

- Automates inference on the test set and saves `inference_results.csv` (predictions and correctness flags).
- Extracts and stores latent embeddings (`latent_representations.npy`) for downstream analysis and visualization.
- Generates annotated evaluation artifacts: confusion matrix PNG, classification_report JSON, learning curves, and t-SNE/UMAP plots.
- Includes an error-analysis workflow that saves spectrogram images for misclassified examples to aid debugging.
- Adds Colab/Drive integration and artifact synchronization for reproducibility (search for latest model, save weights and reports to Drive).

**Conclusion:** The notebook reuses core ideas from mdeff/fma (audio handling and baseline evaluation) but extends them with automation, richer diagnostics, and Colab-oriented tooling.

***
**Purpose:** This notebook is designed to evaluate the trained model and analyze its decision-making process through advanced visualization. It identifies the latest model in `LOCAL_DATA` and performs a complete inference pass on the test set to generate `inference_results.csv`.

The analysis further includes extracting feature embeddings to save as `latent_representations.npy`, alongside computing a detailed confusion matrix and t-SNE plot to visualize genre separation. Finally, the workflow generates a `feature_analysis_meta.json` manifest and synchronizes all generated results back to **Google Drive** for long-term project persistence.

### Workspace Configuration and Environment Setup

The script mounts Google Drive (if running in Colab), defines the primary project directories, creates the necessary local folders, and subsequently switches the current working directory to them.

In [ ]:
# mount Drive (Colab). If not running in Colab, mount will be skipped gracefully.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print("google.colab.drive.mount skipped (not running in Colab or import failed).")

import os, shutil

WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive/waveform_analysis_outputs"

# ensure WORKDIR and LOCAL_DATA exist
os.makedirs(LOCAL_DATA, exist_ok=True)

# change working directory
os.chdir(WORKDIR)

print("WORKDIR:", WORKDIR)
print("LOCAL_DATA:", LOCAL_DATA)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("WORKDIR contents (first 20):", os.listdir(WORKDIR)[:20])

This code synchronizes data from Google Drive to the local Colab environment by calculating the total file count, copying only missing items, and displaying real-time progress while tracking successful transfers, skips, and errors.

In [ ]:
import os, shutil, sys

# Define Source (Drive) and Destination (Local Colab) paths
SRC = "/content/drive/MyDrive/waveform_analysis_outputs/data_storage"
DST = "/content/waveform_genre_project/data_storage"

# Create destination directory if it doesn't exist
os.makedirs(DST, exist_ok=True)

# 1) Calculate total number of files for progress tracking
total_files = 0
for _, _, files in os.walk(SRC):
    total_files += len(files)

if total_files == 0:
    print("There are no files to copy in SRC:", SRC)
else:
    copied = 0
    skipped = 0
    errors = []

    processed = 0
    # 2) Traverse and copy files
    for root, dirs, files in os.walk(SRC):
        rel = os.path.relpath(root, SRC)
        target_dir = os.path.join(DST, rel) if rel != "." else DST
        os.makedirs(target_dir, exist_ok=True)
        for f in files:
            processed += 1
            src_file = os.path.join(root, f)
            dst_file = os.path.join(target_dir, f)
            try:
                if not os.path.exists(dst_file):
                    # Only copy if the file does not already exist locally (Incremental sync)
                    shutil.copy2(src_file, dst_file)
                    copied += 1
                else:
                    skipped += 1
            except Exception as e:
                errors.append((src_file, str(e)))

            # 3) Update progress every 20 files or at the end
            if processed % 20 == 0 or processed == total_files:
                pct = int(processed / total_files * 100)
                # Use carriage return (\r) to overwrite the line in the console
                sys.stdout.write(f"\rProcessed: {processed}/{total_files} ({pct}%)  Copied: {copied}  Skipped: {skipped}  Errors: {len(errors)}")
                sys.stdout.flush()


    print()
    # Final summary print
    print("Copying is complete.")
    print(f"Total files: {total_files}, copied new: {copied}, skipped (already available): {skipped}, errors: {len(errors)}")
    # Report first 5 errors if any occurred
    if errors:
        print("The first 5 mistakes (src, error):")
        for e in errors[:5]:
            print(" -", e[0], ":", e[1])

    # Show multiple files in the local target directory
    try:
        sample = os.listdir(DST)[:50]
        print("First files in local folder:", sample)
    except Exception:
        pass

This Bash script performs high-speed data synchronization from Google Drive to Colab using the rsync utility, skipping existing files and providing detailed progress updates while comparing source and destination file counts for final verification.

In [ ]:
# Optimized data copying from Google Drive to the local Colab environment.
# Uses 'rsync' (bash) for maximum speed when handling a large number of files.
# Logic: Skips already existing files (--ignore-existing) and copies only the missing ones.
# Advantages: Built-in progress indicator (--info=progress2) and significantly higher
%%bash
# set paths
SRC="/content/drive/MyDrive/waveform_analysis_outputs/data_storage"
DST="/content/waveform_genre_project/data_storage"

# creates a target folder if it does not exist
mkdir -p "$DST"

echo "Source: $SRC"
echo "Destination: $DST"
echo

# check for rsync
if command -v rsync >/dev/null 2>&1; then
  echo "Using rsync (fast). This will SKIP existing files."
  # -a  : archive (recursive, preserves attributes)
  # --ignore-existing : does not overwrite existing files
  # --info=progress2 : shows progress (overall progress)
  # trailing slash on SRC copies contents of SRC into DST
  rsync -a --info=progress2 --ignore-existing "${SRC}/" "${DST}/"
  RET=$?
else
  echo "rsync not found, falling back to cp -nur (may show no detailed progress)."
  # -n: no-clobber (не презаписва), -u: only copy when src newer, -r: recursive
  cp -nur "${SRC}/" "${DST}/"
  RET=$?
fi

echo
if [ $RET -ne 0 ]; then
  echo "Copy command returned non-zero exit code: $RET"
else
  echo "Copy finished successfully."
fi

#summary: number of files in SRC and DST (only files in the root of data_storage)
echo
echo "Summary (top-level):"
echo "SRC top-level files:"
ls -1 "${SRC}" | sed -n '1,50p' || true
echo
echo "DST top-level files:"
ls -1 "${DST}" | sed -n '1,50p' || true

# number of files (optional)
echo
echo "Counting total files (may take a few seconds)..."
SRC_CNT=$(find "${SRC}" -type f | wc -l)
DST_CNT=$(find "${DST}" -type f | wc -l)
echo "Files in SRC: $SRC_CNT"
echo "Files now in DST: $DST_CNT"

In [ ]:
# %%bash
# # Progress-aware copy from Drive to local using rsync (preferred) or cp fallback.
# SRC="/content/drive/MyDrive/waveform_analysis_outputs/data_storage"
# DST="/content/waveform_genre_project/data_storage"
# LOG="/tmp/rsync_copy.log"
# INTERVAL=2   # seconds between progress updates

# mkdir -p "$DST"

# if [ ! -d "$SRC" ]; then
#   echo "Source does not exist: $SRC"
#   exit 1
# fi

# echo "Counting source files and total size (this may take a few seconds)..."
# # total bytes (du is fast); fallback to summing file sizes if you prefer exact file bytes
# TOTAL_BYTES=$(du -sb "$SRC" 2>/dev/null | cut -f1)
# TOTAL_FILES=$(find "$SRC" -type f | wc -l)

# if [ -z "$TOTAL_BYTES" ] || [ "$TOTAL_BYTES" -eq 0 ]; then
#   echo "Source seems empty or du failed. TOTAL_BYTES=$TOTAL_BYTES"
#   exit 1
# fi

# echo "SRC files: $TOTAL_FILES, total size: $TOTAL_BYTES bytes"
# echo "Starting copy..."

# START_TS=$(date +%s)

# if command -v rsync >/dev/null 2>&1; then
#   echo "Using rsync (fast). Output is saved to $LOG"
#   # --info=progress2 gives an overall progress; --ignore-existing avoids overwriting
#   rsync -a --info=progress2 --ignore-existing "${SRC}/" "${DST}/" > "$LOG" 2>&1 &
#   RSYNC_PID=$!
# else
#   echo "rsync not found -> falling back to cp -nur (no detailed rsync log)."
#   # run cp in background and log its output (cp quiet, so log may be empty)
#   (cp -nur "${SRC}/" "${DST}/") > "$LOG" 2>&1 &
#   RSYNC_PID=$!
# fi

# # Monitor loop: every INTERVAL seconds compute current bytes in DST and print stats
# while kill -0 $RSYNC_PID 2>/dev/null; do
#   sleep $INTERVAL
#   DST_BYTES=$(du -sb "$DST" 2>/dev/null | cut -f1)
#   if [ -z "$DST_BYTES" ]; then DST_BYTES=0; fi

#   NOW_TS=$(date +%s)
#   ELAPSED=$((NOW_TS - START_TS))
#   if [ "$ELAPSED" -le 0 ]; then ELAPSED=1; fi

#   # percentage (float)
#   PCT=$(awk -v a="$DST_BYTES" -v b="$TOTAL_BYTES" 'BEGIN{ printf "%.1f", (a/b*100) }')

#   # rate bytes/sec
#   RATE=$(awk -v a="$DST_BYTES" -v t="$ELAPSED" 'BEGIN{ if(t>0) printf "%.1f", a/t; else print 0 }')

#   # remaining bytes & ETA
#   REMAIN=$((TOTAL_BYTES - DST_BYTES))
#   if awk "BEGIN {exit !($RATE > 0)}"; then
#     ETA_SEC=$(awk -v r="$REMAIN" -v rate="$RATE" 'BEGIN{ printf "%.0f", r/rate }')
#   else
#     ETA_SEC=0
#   fi

#   # format ETA as H:MM:SS
#   if [ "$ETA_SEC" -le 0 ]; then ETA_STR="N/A"; else
#     H=$((ETA_SEC/3600))
#     M=$(( (ETA_SEC%3600)/60 ))
#     S=$(( ETA_SEC%60 ))
#     printf -v ETA_STR "%d:%02d:%02d" $H $M $S
#   fi

#   # human-readable sizes for display
#   human() {
#     awk -v b="$1" 'function hum(x){
#       s="B KB MB GB TB"; split(s,arr," ");
#       for(i=1; x>=1024 && i<5; i++) x/=1024;
#       return sprintf("%.2f %s", x, arr[i]);
#     }
#     BEGIN{ print hum(b) }'
#   }
#   DST_H=$(human $DST_BYTES)
#   TOT_H=$(human $TOTAL_BYTES)

#   printf "\r%s / %s  (%.1f%%)  rate=%s B/s  ETA=%s    " "$DST_H" "$TOT_H" "$PCT" "$RATE" "$ETA_STR"
#   # flush
# done

# # Wait for rsync/cp to finish and capture exit code
# wait $RSYNC_PID
# RET=$?

# echo
# echo "Copy process finished with exit code: $RET"

# # Final summary
# FINAL_DST_BYTES=$(du -sb "$DST" 2>/dev/null | cut -f1)
# FINAL_DST_FILES=$(find "$DST" -type f | wc -l)
# ELAPSED_TOTAL=$(( $(date +%s) - START_TS ))

# echo "Final: $FINAL_DST_FILES files, $(awk -v b="$FINAL_DST_BYTES" 'function hum(x){
#   s="B KB MB GB TB"; split(s,arr," ");
#   for(i=1; x>=1024 && i<5; i++) x/=1024;
#   return sprintf("%.2f %s", x, arr[i]);
# } BEGIN{ print hum(b) }' ) copied locally."
# echo "Elapsed time: ${ELAPSED_TOTAL}s"
# echo "Log tail (last 20 lines from $LOG):"
# tail -n 20 "$LOG" || true

## Imports and helper load_mel
### Audio Preprocessing for AI

This code prepares an audio file for machine learning models. Here are the 3 main steps it performs:

1. **Loading and Clipping:** It locates the song using its `track_id` and extracts only the first 10 seconds. If the song is shorter, it adds silence (padding with zeros) to ensure all recordings have the same length.
2. **"Audio to Image" Conversion (Spectrogram):** Since computers process images more efficiently, the audio is converted into a **Mel Spectrogram**. This is a visual representation of sound that shows the intensity of different frequencies over time.
3. **Cleaning (Normalization):** The values are converted to decibels and normalized (centered) so that the data is in a standard format for processing.

**Result:** The function returns a digital matrix (a "snapshot" of the sound) ready to be fed into a neural network for tasks like genre recognition, instrument detection, or other audio analysis.

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.manifold import TSNE
import json, glob, time
import librosa

def load_mel_from_track(track_id, SAMPLE_RATE=22050, CLIP_SECONDS=10, N_MELS=128, N_FFT=2048, HOP_LENGTH=512):
    # format track_id as a 6-digit string with leading zeros (e.g., 42 -> "000042")
    s = f"{int(track_id):06d}"
    # construct the file path
    path = os.path.join(LOCAL_DATA, "audio", s[:3], s + ".mp3")
    # load the first 10 seconds of the audio file
    y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=CLIP_SECONDS)
    # ensure consistent length: pad with zeros if shorter than 10s, or clip if longer
    target_len = SAMPLE_RATE * CLIP_SECONDS
    y = np.pad(y, (0, max(0, target_len - len(y))))[:target_len]
    # generate the Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    # convert power to decibels (log scale)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # normalize the data (Zero Mean and Unit Variance)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    # return as float32 for model compatibility
    return mel_db.astype(np.float32)

### Find latest model, rebuild model class, load weights

In [ ]:
# locate latest .pt model in LOCAL_DATA
models = sorted(glob.glob(os.path.join(LOCAL_DATA, "*.pt")), key=os.path.getmtime)
if not models:
    raise FileNotFoundError("No .pt models in " + LOCAL_DATA)
model_file = models[-1]
print("Using model:", model_file)

# reconstruct same SmallCNN architecture
import torch.nn as nn
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(64, 8)  # will set correct n_classes later by loading train.csv
    def forward(self,x, return_embedding=False):
        f = self.features(x).flatten(1)
        f = self.drop(f)
        if return_embedding:
            return f
        return self.head(f)

train_df = pd.read_csv(os.path.join(LOCAL_DATA, "train.csv"))
genres = sorted(train_df["genre"].unique())
n_classes = len(genres)
model = SmallCNN(n_classes)
model.head = nn.Linear(64, n_classes)  # adjust final layer
state = torch.load(model_file, map_location="cpu")
model.load_state_dict(state)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
g2i = {g:i for i,g in enumerate(genres)}

### Run inference on test set and save predictions CSV

In [ ]:
# Load test metadata (expects a CSV with columns including 'track_id' and 'genre')
test_df = pd.read_csv(os.path.join(LOCAL_DATA, "test.csv"))
# will collect per-track prediction records
records = []
# Run inference without tracking gradients to save memory and compute
with torch.no_grad():
    for idx, row in test_df.iterrows():
        # Read track id and ensure it's an integer
        tid = int(row["track_id"])
        # Load precomputed Mel-spectrogram for this track (numpy array, shape: [n_mels, time])
        mel = load_mel_from_track(tid)
        # Convert to a 4D tensor with batch and channel dims: (1, 1, n_mels, time)
        x = torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).to(device)
        # Forward pass through the model -> logits of shape (1, n_classes)
        logits = model(x)
        # Predicted class index (move to CPU and get Python int)
        pred = logits.argmax(1).cpu().item()
        # Map predicted index to genre name
        pred_genre = genres[pred]
        # Record track id, true genre, predicted genre and correctness flag (0/1)
        records.append({"track_id": tid, "true_genre": row["genre"], "pred_genre": pred_genre, "correct": int(pred_genre == row["genre"])})
# Convert collected records to a DataFrame and save results to CSV
preds_df = pd.DataFrame.from_records(records)
preds_csv = os.path.join(LOCAL_DATA, "inference_results.csv")
preds_df.to_csv(preds_csv, index=False)
# Print where predictions were saved and the overall accuracy
print("Saved predictions:", preds_csv)
print("Accuracy:", preds_df["correct"].mean())


This code evaluates the trained model on a new test set. Here is a summary of the process:

1. **Prediction:** For every song in the test list, the model predicts its most likely genre.
2. **Comparison:** The system compares the predicted genre with the actual one and saves the results into a new file (`inference_results.csv`).
3. **Evaluation:** It calculates the overall accuracy of the model (which is 0.325 or 32.5% in this case).